# Pipeline Walkthrough: `build_model()` Step by Step

This notebook unrolls every step of `gbp.build.pipeline.build_model()` into
a separate cell.  Each step shows its inputs, outputs, and the shape of the
resulting DataFrames.  The goal is to build intuition about data
transformations at every stage of the build pipeline.

**Pipeline:**  
RawModelData → validate → time resolution → registry resolution → edges →
lead times → transformations → fleet capacity → assemble ResolvedModelData →
spine assembly

In [1]:
"""Cell 0 — Setup."""
import sys
from pathlib import Path

# Ensure the project root is on sys.path
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from IPython.display import display

In [2]:
"""Cell 1 — Build toy RawModelData.

Small inline dataset: depot d1, stations s1/s2, 3-day horizon,
demand at stations, supply at depot, edges, fleet, and one custom
registry attribute (facility holding cost).
"""
from gbp.core.model import RawModelData
from gbp.core.attributes.registry import AttributeRegistry
from gbp.core.enums import AttributeKind


def make_toy_raw() -> RawModelData:
    """Build a minimal RawModelData entirely inline."""

    facilities = pd.DataFrame([
        {"facility_id": "d1", "facility_type": "depot",   "name": "Depot 1"},
        {"facility_id": "s1", "facility_type": "station", "name": "Station 1"},
        {"facility_id": "s2", "facility_type": "station", "name": "Station 2"},
    ])

    commodity_categories = pd.DataFrame([
        {"commodity_category_id": "bike", "name": "Bicycle", "unit": "units"},
    ])

    resource_categories = pd.DataFrame([
        {
            "resource_category_id": "van",
            "name": "Van",
            "base_capacity": 10.0,
            "capacity_unit": "units",
        },
    ])

    planning_horizon = pd.DataFrame([
        {
            "planning_horizon_id": "ph1",
            "name": "3-day test",
            "start_date": "2025-01-01",
            "end_date": "2025-01-04",
        },
    ])

    planning_horizon_segments = pd.DataFrame([
        {
            "planning_horizon_id": "ph1",
            "segment_index": 0,
            "start_date": "2025-01-01",
            "end_date": "2025-01-04",
            "period_type": "day",
        },
    ])

    periods = pd.DataFrame([
        {
            "period_id": "p0", "planning_horizon_id": "ph1",
            "segment_index": 0, "period_index": 0, "period_type": "day",
            "start_date": "2025-01-01", "end_date": "2025-01-02",
        },
        {
            "period_id": "p1", "planning_horizon_id": "ph1",
            "segment_index": 0, "period_index": 1, "period_type": "day",
            "start_date": "2025-01-02", "end_date": "2025-01-03",
        },
        {
            "period_id": "p2", "planning_horizon_id": "ph1",
            "segment_index": 0, "period_index": 2, "period_type": "day",
            "start_date": "2025-01-03", "end_date": "2025-01-04",
        },
    ])

    facility_roles = pd.DataFrame([
        {"facility_id": "d1", "role": "source"},
        {"facility_id": "d1", "role": "storage"},
        {"facility_id": "s1", "role": "sink"},
        {"facility_id": "s1", "role": "storage"},
        {"facility_id": "s2", "role": "sink"},
        {"facility_id": "s2", "role": "storage"},
    ])

    facility_operations = pd.DataFrame([
        {"facility_id": "d1", "operation_type": "dispatch"},
        {"facility_id": "d1", "operation_type": "storage"},
        {"facility_id": "s1", "operation_type": "receiving"},
        {"facility_id": "s1", "operation_type": "storage"},
        {"facility_id": "s2", "operation_type": "receiving"},
        {"facility_id": "s2", "operation_type": "storage"},
    ])

    edge_rules = pd.DataFrame([
        {
            "source_type": "depot",
            "target_type": "station",
            "commodity_category": "bike",
            "modal_type": "road",
            "enabled": True,
        },
    ])

    # Pre-built edges (so we skip build_edges generation)
    edges = pd.DataFrame([
        {
            "source_id": "d1", "target_id": "s1", "modal_type": "road",
            "distance": 5.0, "distance_unit": "km",
            "lead_time_hours": 12.0, "reliability": 0.95,
        },
        {
            "source_id": "d1", "target_id": "s2", "modal_type": "road",
            "distance": 8.0, "distance_unit": "km",
            "lead_time_hours": 36.0, "reliability": 0.90,
        },
    ])

    edge_commodities = pd.DataFrame([
        {
            "source_id": "d1", "target_id": "s1", "modal_type": "road",
            "commodity_category": "bike", "enabled": True,
            "capacity_consumption": 1.0,
        },
        {
            "source_id": "d1", "target_id": "s2", "modal_type": "road",
            "commodity_category": "bike", "enabled": True,
            "capacity_consumption": 1.0,
        },
    ])

    # Demand: stations need bikes each day
    demand = pd.DataFrame([
        {"facility_id": "s1", "commodity_category": "bike",
         "date": "2025-01-01", "quantity": 3.0, "quantity_unit": "units"},
        {"facility_id": "s1", "commodity_category": "bike",
         "date": "2025-01-02", "quantity": 5.0, "quantity_unit": "units"},
        {"facility_id": "s1", "commodity_category": "bike",
         "date": "2025-01-03", "quantity": 4.0, "quantity_unit": "units"},
        {"facility_id": "s2", "commodity_category": "bike",
         "date": "2025-01-01", "quantity": 2.0, "quantity_unit": "units"},
        {"facility_id": "s2", "commodity_category": "bike",
         "date": "2025-01-02", "quantity": 6.0, "quantity_unit": "units"},
        {"facility_id": "s2", "commodity_category": "bike",
         "date": "2025-01-03", "quantity": 3.0, "quantity_unit": "units"},
    ])

    # Supply: depot produces bikes
    supply = pd.DataFrame([
        {"facility_id": "d1", "commodity_category": "bike",
         "date": "2025-01-01", "quantity": 10.0, "quantity_unit": "units"},
        {"facility_id": "d1", "commodity_category": "bike",
         "date": "2025-01-02", "quantity": 10.0, "quantity_unit": "units"},
        {"facility_id": "d1", "commodity_category": "bike",
         "date": "2025-01-03", "quantity": 10.0, "quantity_unit": "units"},
    ])

    # Initial inventory at stations
    inventory_initial = pd.DataFrame([
        {"facility_id": "d1", "commodity_category": "bike",
         "quantity": 20.0, "quantity_unit": "units"},
        {"facility_id": "s1", "commodity_category": "bike",
         "quantity": 5.0, "quantity_unit": "units"},
        {"facility_id": "s2", "commodity_category": "bike",
         "quantity": 3.0, "quantity_unit": "units"},
    ])

    # Fleet: vans at depot
    resource_fleet = pd.DataFrame([
        {"facility_id": "d1", "resource_category": "van", "count": 2},
    ])

    # Resource compatibility
    resource_commodity_compat = pd.DataFrame([
        {"resource_category": "van", "commodity_category": "bike", "enabled": True},
    ])
    resource_modal_compat = pd.DataFrame([
        {"resource_category": "van", "modal_type": "road", "enabled": True},
    ])

    # Custom registry attribute: daily holding cost per facility
    registry = AttributeRegistry()
    holding_cost_data = pd.DataFrame([
        {"facility_id": "d1", "date": "2025-01-01", "holding_cost": 0.5},
        {"facility_id": "d1", "date": "2025-01-02", "holding_cost": 0.5},
        {"facility_id": "d1", "date": "2025-01-03", "holding_cost": 0.6},
        {"facility_id": "s1", "date": "2025-01-01", "holding_cost": 1.0},
        {"facility_id": "s1", "date": "2025-01-02", "holding_cost": 1.0},
        {"facility_id": "s1", "date": "2025-01-03", "holding_cost": 1.2},
        {"facility_id": "s2", "date": "2025-01-01", "holding_cost": 0.8},
        {"facility_id": "s2", "date": "2025-01-02", "holding_cost": 0.8},
        {"facility_id": "s2", "date": "2025-01-03", "holding_cost": 0.9},
    ])
    registry.register(
        "holding_cost",
        holding_cost_data,
        entity_type="facility",
        kind=AttributeKind.COST,
        grain=("facility_id", "date"),
        value_column="holding_cost",
        aggregation="mean",
        unit="usd_per_unit_per_period",
    )

    return RawModelData(
        facilities=facilities,
        commodity_categories=commodity_categories,
        resource_categories=resource_categories,
        planning_horizon=planning_horizon,
        planning_horizon_segments=planning_horizon_segments,
        periods=periods,
        facility_roles=facility_roles,
        facility_operations=facility_operations,
        edge_rules=edge_rules,
        edges=edges,
        edge_commodities=edge_commodities,
        demand=demand,
        supply=supply,
        inventory_initial=inventory_initial,
        resource_fleet=resource_fleet,
        resource_commodity_compatibility=resource_commodity_compat,
        resource_modal_compatibility=resource_modal_compat,
        attributes=registry,
    )


raw = make_toy_raw()
print(raw.table_summary())

RawModelData — table summary

  entity
  ──────
    facilities: 3 rows (required)
    commodity_categories: 1 rows (required)
    resource_categories: 1 rows (required)
    commodities: —
    resources: —

  temporal
  ────────
    planning_horizon: 1 rows (required)
    planning_horizon_segments: 1 rows (required)
    periods: 3 rows (required)

  behavior
  ────────
    facility_roles: 6 rows (required)
    facility_operations: 6 rows (required)
    facility_availability: —
    edge_rules: 1 rows (required)

  edge
  ────
    edges: 2 rows
    edge_commodities: 2 rows
    edge_capacities: —
    edge_commodity_capacities: —
    edge_vehicles: —

  flow_data
  ─────────
    demand: 6 rows
    supply: 3 rows
    inventory_initial: 3 rows
    inventory_in_transit: —

  transformation
  ──────────────
    transformations: —
    transformation_inputs: —
    transformation_outputs: —

  resource
  ────────
    resource_commodity_compatibility: 1 rows
    resource_modal_compatibility: 1 rows

## Step 1 — Validation

First pipeline step: validate referential integrity, unit consistency,
temporal coverage, and graph connectivity.  Errors block the build;
warnings are informational.

In [3]:
"""Cell 2 — Validation."""
from gbp.build.validation import validate_raw_model

result = validate_raw_model(raw)

print(f"is_valid: {result.is_valid}")
print(f"has_warnings: {result.has_warnings}")
if result.errors:
    print("\nIssues:")
    for e in result.errors:
        print(f"  [{e.level}] {e.category}/{e.entity}: {e.message}")
else:
    print("\nNo issues found — model is clean.")

is_valid: True
has_warnings: False

No issues found — model is clean.


## Step 2 — Time Resolution

Maps raw `date` columns to `period_id` buckets.  For each time-varying
structural table (demand, supply, edge_capacities, etc.), rows with dates
falling within a period's `[start_date, end_date)` are aggregated.

Since our periods are daily and each date maps 1:1 to a period, the
aggregation is trivial here — but with weekly/monthly periods it would
sum or average across multiple dates.

In [4]:
"""Cell 3 — Time resolution of structural tables."""
from gbp.build.time_resolution import resolve_all_time_varying

periods = raw.periods.copy()
resolved_time = resolve_all_time_varying(raw, periods)

print("Periods grid:")
display(periods)

print(f"\nResolved tables: {list(resolved_time.keys())}")

print("\n--- Raw demand (date-based) ---")
display(raw.demand)

print("\n--- Resolved demand (period_id-based) ---")
display(resolved_time["demand"])

print("\n--- Raw supply (date-based) ---")
display(raw.supply)

print("\n--- Resolved supply (period_id-based) ---")
display(resolved_time["supply"])

Periods grid:


,period_id,planning_horizon_id,segment_index,period_index,period_type,start_date,end_date
0,p0,ph1,0,0,day,2025-01-01,2025-01-02
1,p1,ph1,0,1,day,2025-01-02,2025-01-03
2,p2,ph1,0,2,day,2025-01-03,2025-01-04



Resolved tables: ['demand', 'supply']

--- Raw demand (date-based) ---


,facility_id,commodity_category,date,quantity,quantity_unit
0,s1,bike,2025-01-01,3.0,units
1,s1,bike,2025-01-02,5.0,units
2,s1,bike,2025-01-03,4.0,units
3,s2,bike,2025-01-01,2.0,units
4,s2,bike,2025-01-02,6.0,units
5,s2,bike,2025-01-03,3.0,units



--- Resolved demand (period_id-based) ---


,facility_id,commodity_category,period_id,quantity
0,s1,bike,p0,3.0
1,s1,bike,p1,5.0
2,s1,bike,p2,4.0
3,s2,bike,p0,2.0
4,s2,bike,p1,6.0
5,s2,bike,p2,3.0



--- Raw supply (date-based) ---


,facility_id,commodity_category,date,quantity,quantity_unit
0,d1,bike,2025-01-01,10.0,units
1,d1,bike,2025-01-02,10.0,units
2,d1,bike,2025-01-03,10.0,units



--- Resolved supply (period_id-based) ---


,facility_id,commodity_category,period_id,quantity
0,d1,bike,p0,10.0
1,d1,bike,p1,10.0
2,d1,bike,p2,10.0


## Step 3 — Registry Attribute Resolution

Same `date → period_id` mapping, but applied to the `AttributeRegistry`.
Each time-varying registered attribute gets its `date` column replaced
by `period_id`, with values aggregated per the attribute's `aggregation`
function.

In [5]:
"""Cell 4 — Registry attribute resolution."""
from gbp.build.time_resolution import resolve_registry_attributes

resolved_attrs = resolve_registry_attributes(raw.attributes, periods)

print("--- Raw registry summary ---")
print(raw.attributes.summary())

print("\n--- Raw holding_cost data (date-based) ---")
display(raw.attributes.get("holding_cost").data)

print("\n--- Resolved registry summary ---")
print(resolved_attrs.summary())

print("\n--- Resolved holding_cost data (period_id-based) ---")
display(resolved_attrs.get("holding_cost").data)

--- Raw registry summary ---
    holding_cost: 9 rows  facility  COST  [facility_id × date]

--- Raw holding_cost data (date-based) ---


,facility_id,date,holding_cost
0,d1,2025-01-01,0.5
1,d1,2025-01-02,0.5
2,d1,2025-01-03,0.6
3,s1,2025-01-01,1.0
4,s1,2025-01-02,1.0
5,s1,2025-01-03,1.2
6,s2,2025-01-01,0.8
7,s2,2025-01-02,0.8
8,s2,2025-01-03,0.9



--- Resolved registry summary ---
    holding_cost: 9 rows  facility  COST  [facility_id × date]

--- Resolved holding_cost data (period_id-based) ---


,facility_id,period_id,holding_cost
0,d1,p0,0.5
1,d1,p1,0.5
2,d1,p2,0.6
3,s1,p0,1.0
4,s1,p1,1.0
5,s1,p2,1.2
6,s2,p0,0.8
7,s2,p1,0.8
8,s2,p2,0.9


## Step 4 — Edges

If `raw.edges` is already populated, the pipeline uses them directly.
Otherwise, `build_edges()` generates edges from `edge_rules` via a
cross-join of facilities matching `(source_type, target_type)`.

Our toy model has pre-built edges, so we just display them.

In [6]:
"""Cell 5 — Edges."""
from gbp.build.pipeline import _ensure_edges_and_commodities

edges_df, ec_df = _ensure_edges_and_commodities(raw)

print("--- Edges (pre-built in our toy model) ---")
display(edges_df)

print("\n--- Edge commodities ---")
display(ec_df)

print(f"\nNote: since raw.edges was populated, build_edges() was skipped.")
print(f"If edges were None, build_edges() would generate them from edge_rules:")
display(raw.edge_rules)

--- Edges (pre-built in our toy model) ---


,source_id,target_id,modal_type,distance,distance_unit,lead_time_hours,reliability
0,d1,s1,road,5.0,km,12.0,0.95
1,d1,s2,road,8.0,km,36.0,0.90



--- Edge commodities ---


,source_id,target_id,modal_type,commodity_category,enabled,capacity_consumption
0,d1,s1,road,bike,True,1.0
1,d1,s2,road,bike,True,1.0



Note: since raw.edges was populated, build_edges() was skipped.
If edges were None, build_edges() would generate them from edge_rules:


,source_type,target_type,commodity_category,modal_type,enabled
0,depot,station,bike,road,True


## Step 5 — Lead Time Resolution

`resolve_lead_times()` converts `lead_time_hours` into integer
`lead_time_periods` using `ceil(hours / period_duration)`.  It produces
one row per `(edge × departure period)` with the `arrival_period_id`.

If the arrival falls beyond the planning horizon, `arrival_period_id` is
`NaN` — the shipment departs but never arrives within the model window.

In [7]:
"""Cell 6 — Lead time resolution."""
from gbp.build.lead_time import resolve_lead_times

edge_lead_time_resolved = resolve_lead_times(edges_df, periods)

print("--- Edge lead time resolved ---")
print(f"Shape: {edge_lead_time_resolved.shape}")
print(f"Columns: {list(edge_lead_time_resolved.columns)}")
display(edge_lead_time_resolved)

# Highlight NaN arrivals (beyond horizon)
beyond = edge_lead_time_resolved[
    edge_lead_time_resolved["arrival_period_id"].isna()
]
if not beyond.empty:
    print("\nRows with arrival beyond planning horizon (NaN arrival_period_id):")
    display(beyond)
else:
    print("\nAll shipments arrive within the planning horizon.")

--- Edge lead time resolved ---
Shape: (6, 6)
Columns: ['source_id', 'target_id', 'modal_type', 'period_id', 'lead_time_periods', 'arrival_period_id']


,source_id,target_id,modal_type,period_id,lead_time_periods,arrival_period_id
0,d1,s1,road,p0,1,p1
1,d1,s1,road,p1,1,p2
2,d1,s1,road,p2,1,NaN
3,d1,s2,road,p0,2,p2
4,d1,s2,road,p1,2,NaN
5,d1,s2,road,p2,2,NaN



Rows with arrival beyond planning horizon (NaN arrival_period_id):


,source_id,target_id,modal_type,period_id,lead_time_periods,arrival_period_id
2,d1,s1,road,p2,1,NaN
4,d1,s2,road,p1,2,NaN
5,d1,s2,road,p2,2,NaN


## Step 6 — Transformation Resolution

`resolve_transformations()` joins transformation definitions with
facilities to create a denormalized view of N:M commodity conversions.

Our toy model has no transformations, so this returns `None`.

In [8]:
"""Cell 7 — Transformation resolution."""
from gbp.build.transformation import resolve_transformations

transformation_resolved = resolve_transformations(
    raw.facilities,
    raw.transformations,
    raw.transformation_inputs,
    raw.transformation_outputs,
)

print(f"transformation_resolved: {transformation_resolved}")
print(
    "\nExpected: None — our toy model has no transformations."
    "\nIn a model with N:M conversions (e.g. raw materials → finished goods),"
    "\nthis would be a DataFrame with one row per facility × transformation × IO pair."
)

transformation_resolved: None

Expected: None — our toy model has no transformations.
In a model with N:M conversions (e.g. raw materials → finished goods),
this would be a DataFrame with one row per facility × transformation × IO pair.


## Step 7 — Fleet Capacity

`compute_fleet_capacity()` calculates `effective_capacity` per
`(facility, resource_category)` as `count × base_capacity` from
the fleet table joined with resource categories.

In [9]:
"""Cell 8 — Fleet capacity."""
from gbp.build.fleet_capacity import compute_fleet_capacity

fleet_capacity = compute_fleet_capacity(
    raw.resource_fleet,
    raw.resource_categories,
    raw.resources,
)

print("--- Resource fleet (input) ---")
display(raw.resource_fleet)

print("\n--- Resource categories (base_capacity) ---")
display(raw.resource_categories)

print("\n--- Fleet capacity (computed: count × base_capacity) ---")
display(fleet_capacity)

--- Resource fleet (input) ---


,facility_id,resource_category,count
0,d1,van,2



--- Resource categories (base_capacity) ---


,resource_category_id,name,base_capacity,capacity_unit
0,van,Van,10.0,units



--- Fleet capacity (computed: count × base_capacity) ---


,facility_id,resource_category,effective_capacity
0,d1,van,20.0


## Step 8 — Assemble ResolvedModelData

All resolved tables are combined into a single `ResolvedModelData`
instance via `ResolvedModelData.from_raw()`.  This factory method
prefers time-resolved tables when available, otherwise passes the
raw table through.

In [10]:
"""Cell 9 — Assemble ResolvedModelData."""
from gbp.core.model import ResolvedModelData

resolved = ResolvedModelData.from_raw(
    raw,
    periods=periods,
    resolved_time=resolved_time,
    resolved_attrs=resolved_attrs,
    edges=edges_df,
    edge_commodities=ec_df,
    edge_lead_time_resolved=(
        edge_lead_time_resolved if not edge_lead_time_resolved.empty else None
    ),
    transformation_resolved=transformation_resolved,
    fleet_capacity=fleet_capacity,
)

print(resolved.table_summary())

ResolvedModelData — table summary

  entity
  ──────
    facilities: 3 rows (required)
    commodity_categories: 1 rows (required)
    resource_categories: 1 rows (required)
    commodities: —
    resources: —

  temporal
  ────────
    planning_horizon: 1 rows (required)
    planning_horizon_segments: 1 rows (required)
    periods: 3 rows (required)

  behavior
  ────────
    facility_roles: 6 rows (required)
    facility_operations: 6 rows (required)
    facility_availability: —
    edge_rules: 1 rows (required)

  edge
  ────
    edges: 2 rows
    edge_commodities: 2 rows
    edge_capacities: —
    edge_commodity_capacities: —
    edge_vehicles: —

  flow_data
  ─────────
    demand: 6 rows
    supply: 3 rows
    inventory_initial: 3 rows
    inventory_in_transit: —

  transformation
  ──────────────
    transformations: —
    transformation_inputs: —
    transformation_outputs: —

  resource
  ────────
    resource_commodity_compatibility: 1 rows
    resource_modal_compatibility: 1

## Step 9 — Spine Assembly

`assemble_spines()` groups registered attributes by entity type and
grain, then merges them onto base entity tables.  The result is
per-entity dicts of `{grain_group_name: DataFrame}` — ready for
vectorized lookups by consumers (Environment, Optimizer, Analytics).

In [11]:
"""Cell 10 — Spine assembly."""
from gbp.build.spine import assemble_spines

spines = assemble_spines(resolved)

# Attach spines to resolved (same as build_model does)
resolved.facility_spines = spines["facility"] or None
resolved.edge_spines = spines["edge"] or None
resolved.resource_spines = spines["resource"] or None

for entity, spine_dict in spines.items():
    if spine_dict:
        print(f"\n=== {entity} spines ===")
        print(f"  Grain groups: {list(spine_dict.keys())}")
        for group_name, df in spine_dict.items():
            print(f"\n  --- {group_name} ({df.shape[0]} rows × {df.shape[1]} cols) ---")
            display(df.head())
    else:
        print(f"\n=== {entity} spines: (empty) ===")


=== facility spines ===
  Grain groups: ['group_0']

  --- group_0 (9 rows × 5 cols) ---


,facility_id,facility_type,name,period_id,holding_cost
0,d1,depot,Depot 1,p0,0.5
1,d1,depot,Depot 1,p1,0.5
2,d1,depot,Depot 1,p2,0.6
3,s1,station,Station 1,p0,1.0
4,s1,station,Station 1,p1,1.0



=== edge spines ===
  Grain groups: ['group_0']

  --- group_0 (2 rows × 13 cols) ---


,source_id,target_id,modal_type,distance,distance_unit,lead_time_hours,reliability,edge_distance_x,edge_distance_y,edge_lead_time_hours_x,edge_lead_time_hours_y,edge_reliability_x,edge_reliability_y
0,d1,s1,road,5.0,km,12.0,0.95,5.0,5.0,12.0,12.0,0.95,0.95
1,d1,s2,road,8.0,km,36.0,0.90,8.0,8.0,36.0,36.0,0.90,0.90



=== resource spines ===
  Grain groups: ['group_0']

  --- group_0 (1 rows × 6 cols) ---


,resource_category,name,base_capacity,capacity_unit,resource_base_capacity_x,resource_base_capacity_y
0,van,Van,10.0,units,10.0,10.0


## Sanity Check — Compare with `build_model()`

Run the full pipeline as a black box on a fresh copy of the raw data
and verify that the shapes of key tables match our manual assembly.

In [12]:
"""Cell 11 — Sanity: compare with build_model()."""
from gbp.build.pipeline import build_model

# Fresh raw model (registry is mutable, so we rebuild)
resolved_auto = build_model(make_toy_raw())

# Compare key table shapes
checks = [
    "facilities", "periods", "demand", "supply",
    "edges", "edge_commodities", "edge_lead_time_resolved",
    "fleet_capacity", "inventory_initial",
]

print(f"{'Table':<30} {'Manual':>10} {'build_model()':>14}  Match")
print("-" * 65)
all_ok = True
for name in checks:
    manual_df = getattr(resolved, name, None)
    auto_df = getattr(resolved_auto, name, None)
    m_shape = manual_df.shape if manual_df is not None else None
    a_shape = auto_df.shape if auto_df is not None else None
    ok = m_shape == a_shape
    if not ok:
        all_ok = False
    print(f"{name:<30} {str(m_shape):>10} {str(a_shape):>14}  {'OK' if ok else 'MISMATCH'}")

# Compare spine keys
for entity in ["facility", "edge", "resource"]:
    manual_sp = getattr(resolved, f"{entity}_spines", None) or {}
    auto_sp = getattr(resolved_auto, f"{entity}_spines", None) or {}
    ok = set(manual_sp.keys()) == set(auto_sp.keys())
    if not ok:
        all_ok = False
    print(
        f"{entity}_spines keys          "
        f"{str(sorted(manual_sp.keys())):>10} "
        f"{str(sorted(auto_sp.keys())):>14}  "
        f"{'OK' if ok else 'MISMATCH'}"
    )

print(f"\n{'All checks passed!' if all_ok else 'Some mismatches — investigate above.'}")

Table                              Manual  build_model()  Match
-----------------------------------------------------------------
facilities                         (3, 3)         (3, 3)  OK
periods                            (3, 7)         (3, 7)  OK
demand                             (6, 4)         (6, 4)  OK
supply                             (3, 4)         (3, 4)  OK
edges                              (2, 7)         (2, 7)  OK
edge_commodities                   (2, 6)         (2, 6)  OK
edge_lead_time_resolved            (6, 6)         (6, 6)  OK
fleet_capacity                     (1, 3)         (1, 3)  OK
inventory_initial                  (3, 4)         (3, 4)  OK
facility_spines keys          ['group_0']    ['group_0']  OK
edge_spines keys          ['group_0']    ['group_0']  OK
resource_spines keys          ['group_0']    ['group_0']  OK

All checks passed!
